```
# Lab type: debug
# Course: ML402 — Reinforcement Learning
# Lesson: Deep Q-Networks
# Task: The DQN implementation below has 3 bugs. All run without error but each
#       defeats one of DQN's stability mechanisms. Find each bug, explain it,
#       and write the corrected version in the fix cell.
```

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import gymnasium as gym

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ── Shared network definition ────────────────────────────────────────────────

class QNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)


# ── Replay buffer ────────────────────────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = zip(*batch)
        return (
            torch.tensor(np.array(s),      dtype=torch.float32),
            torch.tensor(a,                dtype=torch.long),
            torch.tensor(r,                dtype=torch.float32),
            torch.tensor(np.array(s_next), dtype=torch.float32),
            torch.tensor(done,             dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


# ── Reference hyperparameters ─────────────────────────────────────────────────

GAMMA               = 0.99
LR                  = 1e-3
BATCH_SIZE          = 64
BUFFER_CAPACITY     = 10_000
MIN_BUFFER_SIZE     = 1_000
TARGET_UPDATE_FREQ  = 10
EPSILON_START       = 1.0
EPSILON_END         = 0.01
EPSILON_DECAY_EPS   = 200
MAX_EPISODES        = 300

print("Setup complete. QNet and ReplayBuffer defined.")


## Bug 1: Online network used for TD target

The training step below computes the TD target using the **wrong network**. DQN's target-network mechanism exists specifically to prevent this. The code trains and the reward improves — but one of the two core stability mechanisms is entirely absent.

Run the cell and note the reward curve. Then identify which line contains the bug.

In [ ]:
# --- BUGGY CODE (Bug 1) ---

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net      = QNet(state_dim, n_actions)
target_net = QNet(state_dim, n_actions)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=LR)
loss_fn   = nn.MSELoss()
buffer    = ReplayBuffer(BUFFER_CAPACITY)

episode_rewards = []

for episode in range(MAX_EPISODES):
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPS)
    s, _ = env.reset()
    total_reward = 0.0
    done = False

    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer.push(s, a, r, s_next, float(done))
        s = s_next
        total_reward += r

        if len(buffer) >= MIN_BUFFER_SIZE:
            states, actions, rewards, next_states, dones = buffer.sample(BATCH_SIZE)
            q_pred = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next   = q_net(next_states).max(dim=1).values   # BUG: q_net, not target_net
                q_target = rewards + GAMMA * q_next * (1.0 - dones)
            loss = loss_fn(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    episode_rewards.append(total_reward)
    if episode % TARGET_UPDATE_FREQ == 0:
        target_net.load_state_dict(q_net.state_dict())
    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1:>4d}  |  avg reward (last 50): {avg:.1f}  |  ε={epsilon:.3f}")

env.close()


**Explain the bug:** DQN uses a target network to compute the TD target `y = r + γ·max Q_{θ⁻}(s',a')`. What happens when `q_net` is used instead of `target_net` here? Why might training still converge on CartPole while failing on harder environments with sparse rewards?

*(Write your answer here.)*

In [ ]:
# Fix for Bug 1: use target_net when computing the TD target

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net_f1      = QNet(state_dim, n_actions)
target_net_f1 = QNet(state_dim, n_actions)
target_net_f1.load_state_dict(q_net_f1.state_dict())
target_net_f1.eval()

optimizer_f1 = optim.Adam(q_net_f1.parameters(), lr=LR)
loss_fn_f1   = nn.MSELoss()
buffer_f1    = ReplayBuffer(BUFFER_CAPACITY)

episode_rewards_f1 = []

for episode in range(MAX_EPISODES):
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPS)
    s, _ = env.reset()
    total_reward = 0.0
    done = False

    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net_f1(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer_f1.push(s, a, r, s_next, float(done))
        s = s_next
        total_reward += r

        if len(buffer_f1) >= MIN_BUFFER_SIZE:
            states, actions, rewards, next_states, dones = buffer_f1.sample(BATCH_SIZE)
            q_pred = q_net_f1(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next   = target_net_f1(next_states).max(dim=1).values  # FIX: target_net
                q_target = rewards + GAMMA * q_next * (1.0 - dones)
            loss = loss_fn_f1(q_pred, q_target)
            optimizer_f1.zero_grad()
            loss.backward()
            optimizer_f1.step()

    episode_rewards_f1.append(total_reward)
    if episode % TARGET_UPDATE_FREQ == 0:
        target_net_f1.load_state_dict(q_net_f1.state_dict())
    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards_f1[-50:])
        print(f"Episode {episode+1:>4d}  |  avg reward (last 50): {avg:.1f}  |  ε={epsilon:.3f}")

env.close()


## Bug 2: Target network parameters included in the optimiser

The snippet below correctly uses `target_net` for the TD target — but the optimiser receives parameters from **both** networks. As a result, gradient updates from `loss.backward()` also flow into the target network. Run it and compare the reward stability to Bug 1's fix.

In [ ]:
# --- BUGGY CODE (Bug 2) ---

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net_b2      = QNet(state_dim, n_actions)
target_net_b2 = QNet(state_dim, n_actions)
target_net_b2.load_state_dict(q_net_b2.state_dict())
target_net_b2.eval()

# BUG: both networks' parameters passed to the optimiser
optimizer_b2 = optim.Adam(
    list(q_net_b2.parameters()) + list(target_net_b2.parameters()),
    lr=LR
)
loss_fn_b2 = nn.MSELoss()
buffer_b2  = ReplayBuffer(BUFFER_CAPACITY)

episode_rewards_b2 = []

for episode in range(MAX_EPISODES):
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPS)
    s, _ = env.reset()
    total_reward = 0.0
    done = False

    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net_b2(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer_b2.push(s, a, r, s_next, float(done))
        s = s_next
        total_reward += r

        if len(buffer_b2) >= MIN_BUFFER_SIZE:
            states, actions, rewards, next_states, dones = buffer_b2.sample(BATCH_SIZE)
            q_pred = q_net_b2(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next   = target_net_b2(next_states).max(dim=1).values
                q_target = rewards + GAMMA * q_next * (1.0 - dones)
            loss = loss_fn_b2(q_pred, q_target)
            optimizer_b2.zero_grad()
            loss.backward()
            optimizer_b2.step()

    episode_rewards_b2.append(total_reward)
    if episode % TARGET_UPDATE_FREQ == 0:
        target_net_b2.load_state_dict(q_net_b2.state_dict())
    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards_b2[-50:])
        print(f"Episode {episode+1:>4d}  |  avg reward (last 50): {avg:.1f}  |  ε={epsilon:.3f}")

env.close()


**Explain the bug:** The target network should only be updated by the explicit `load_state_dict` call every `TARGET_UPDATE_FREQ` episodes. When its parameters are inside the optimiser, what happens instead? Why does this partially undermine the stability guarantee even though `torch.no_grad()` prevents the loss gradient from flowing through the target network's *forward pass*?

*(Write your answer here.)*

In [ ]:
# Fix for Bug 2: optimiser only receives q_net parameters

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net_f2      = QNet(state_dim, n_actions)
target_net_f2 = QNet(state_dim, n_actions)
target_net_f2.load_state_dict(q_net_f2.state_dict())
target_net_f2.eval()

# FIX: only q_net parameters in the optimiser
optimizer_f2 = optim.Adam(q_net_f2.parameters(), lr=LR)
loss_fn_f2   = nn.MSELoss()
buffer_f2    = ReplayBuffer(BUFFER_CAPACITY)

episode_rewards_f2 = []

for episode in range(MAX_EPISODES):
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPS)
    s, _ = env.reset()
    total_reward = 0.0
    done = False

    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net_f2(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer_f2.push(s, a, r, s_next, float(done))
        s = s_next
        total_reward += r

        if len(buffer_f2) >= MIN_BUFFER_SIZE:
            states, actions, rewards, next_states, dones = buffer_f2.sample(BATCH_SIZE)
            q_pred = q_net_f2(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next   = target_net_f2(next_states).max(dim=1).values
                q_target = rewards + GAMMA * q_next * (1.0 - dones)
            loss = loss_fn_f2(q_pred, q_target)
            optimizer_f2.zero_grad()
            loss.backward()
            optimizer_f2.step()

    episode_rewards_f2.append(total_reward)
    if episode % TARGET_UPDATE_FREQ == 0:
        target_net_f2.load_state_dict(q_net_f2.state_dict())
    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards_f2[-50:])
        print(f"Episode {episode+1:>4d}  |  avg reward (last 50): {avg:.1f}  |  ε={epsilon:.3f}")

env.close()


## Bug 3: Terminal transitions not masked in TD target

The correct TD target for a terminal transition (where `done=True`) is `y = r` — you don't bootstrap from the next state because there is none. The masking `(1 - done)` achieves this.

The snippet below omits the masking term. The code trains, the reward may still climb, but the Q-values for transitions leading to episode termination are systematically overestimated.

In [ ]:
# --- BUGGY CODE (Bug 3) ---
# Only the TD target computation is shown; assume everything else is correct.

# Inside the training step:
# states, actions, rewards, next_states, dones = buffer.sample(BATCH_SIZE)

# q_pred = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
# with torch.no_grad():
#     q_next   = target_net(next_states).max(dim=1).values
#     q_target = rewards + GAMMA * q_next          # BUG: no (1 - dones) masking

# Run the demo to observe the bias in learned Q-values near termination

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net_b3      = QNet(state_dim, n_actions)
target_net_b3 = QNet(state_dim, n_actions)
target_net_b3.load_state_dict(q_net_b3.state_dict())
target_net_b3.eval()

optimizer_b3 = optim.Adam(q_net_b3.parameters(), lr=LR)
loss_fn_b3   = nn.MSELoss()
buffer_b3    = ReplayBuffer(BUFFER_CAPACITY)

for episode in range(MAX_EPISODES):
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPS)
    s, _ = env.reset()
    done = False
    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net_b3(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer_b3.push(s, a, r, s_next, float(done))
        s = s_next

        if len(buffer_b3) >= MIN_BUFFER_SIZE:
            states_b, actions_b, rewards_b, next_states_b, dones_b = buffer_b3.sample(BATCH_SIZE)
            q_pred_b = q_net_b3(states_b).gather(1, actions_b.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next_b   = target_net_b3(next_states_b).max(dim=1).values
                q_target_b = rewards_b + GAMMA * q_next_b  # BUG: no masking
            loss_b = loss_fn_b3(q_pred_b, q_target_b)
            optimizer_b3.zero_grad()
            loss_b.backward()
            optimizer_b3.step()
    if episode % TARGET_UPDATE_FREQ == 0:
        target_net_b3.load_state_dict(q_net_b3.state_dict())

# Sample Q-values for a terminal-like state (pole nearly fallen)
with torch.no_grad():
    terminal_obs = torch.tensor([0.0, 0.0, 0.20, 2.5], dtype=torch.float32)  # near-fall
    q_vals_bug   = q_net_b3(terminal_obs)
print("Bug 3 — Q-values near termination:", q_vals_bug.tolist())

env.close()


**Explain the bug:** When `done=True`, the transition leads to a terminal state with no future return. What does `rewards + GAMMA * q_next` compute for such a transition instead? How does bootstrapping from a non-existent next state affect Q-value learning for the actions that end an episode?

*(Write your answer here.)*

In [ ]:
# Fix for Bug 3: multiply q_next by (1 - dones) to zero out terminal bootstraps

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

q_net_f3      = QNet(state_dim, n_actions)
target_net_f3 = QNet(state_dim, n_actions)
target_net_f3.load_state_dict(q_net_f3.state_dict())
target_net_f3.eval()

optimizer_f3 = optim.Adam(q_net_f3.parameters(), lr=LR)
loss_fn_f3   = nn.MSELoss()
buffer_f3    = ReplayBuffer(BUFFER_CAPACITY)

for episode in range(MAX_EPISODES):
    epsilon = max(EPSILON_END, EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPS)
    s, _ = env.reset()
    done = False
    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net_f3(torch.tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer_f3.push(s, a, r, s_next, float(done))
        s = s_next

        if len(buffer_f3) >= MIN_BUFFER_SIZE:
            states_f, actions_f, rewards_f, next_states_f, dones_f = buffer_f3.sample(BATCH_SIZE)
            q_pred_f = q_net_f3(states_f).gather(1, actions_f.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next_f   = target_net_f3(next_states_f).max(dim=1).values
                q_target_f = rewards_f + GAMMA * q_next_f * (1.0 - dones_f)  # FIX: mask terminals
            loss_f = loss_fn_f3(q_pred_f, q_target_f)
            optimizer_f3.zero_grad()
            loss_f.backward()
            optimizer_f3.step()
    if episode % TARGET_UPDATE_FREQ == 0:
        target_net_f3.load_state_dict(q_net_f3.state_dict())

with torch.no_grad():
    terminal_obs = torch.tensor([0.0, 0.0, 0.20, 2.5], dtype=torch.float32)
    q_vals_fix   = q_net_f3(terminal_obs)
print("Fix 3 — Q-values near termination:", q_vals_fix.tolist())
print("(values should be lower than Bug 3 — less overestimation of terminal states)")

env.close()


## Summary

> **For each bug, write one sentence on what went wrong and which DQN stability mechanism it defeated.**

1.
2.
3.